# Eigenvector Analysis — S&P 500 Correlation Structure
**Author:** Elsa Susana Ochoa  
**Master's Thesis — UNAM, 2018**

Analyzes the eigenvectors of the S&P 500 correlation matrix using two complementary metrics:

- **Participation Ratio (PR)**: how many stocks contribute to each eigenvector mode
- **Overlap Spectral Contribution (OSC)**: how much the Power-Mapped eigenvectors deviate from the original (non-PM) eigenvectors, mode by mode

## 1. Imports and Load Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy import linalg as LA
from scipy import linalg, stats
%matplotlib inline

with open('../data/Symbols.txt', 'r') as S:
    s = [line.split()[0] for line in S]
lensymbols = len(s)

with open('../data/Dates.txt', 'r') as D:
    d = [line.split()[0] for line in D]
lendates = len(d)

with open('../data/Gics.txt', 'r') as G:
    g = [line.split()[0] for line in G]

Prices = np.loadtxt('../data/Prices.txt')

N = 293
T = 44
q = 1.05
numVentana = 262

print(f'Stocks: {lensymbols}, Dates: {lendates}, q={q}')

## 2. Helper Functions

**Participation Ratio (PR)**

$$PR_k = \frac{1}{N \sum_i u_{ki}^4}$$

- PR near 1: all stocks contribute equally (global market mode)
- PR near 0: few stocks dominate (sector or individual mode)

**Overlap Spectral Contribution (OSC)**

Measures how much a Power-Mapped eigenvector $v^{PM}_i$ overlaps with the bulk (noise-like) subspace of the original eigenvectors $v_j$:

$$OSC_i = \sum_{j=N-T+1}^{N} \left(v^{PM}_i \cdot v_j\right)^2$$

Low OSC for the top eigenvectors confirms they carry information independent of the noise subspace.

In [ ]:
def participation_ratio(v):
    """v: eigenvector matrix (N x N), columns are eigenvectors"""
    vPR = np.sum(v**4, axis=0)
    return 1 / (N * vPR)

def overlap_spectral_contribution(v_pm, v_orig, n, t):
    """How much each Power-Mapped eigenvector overlaps with the
    noise subspace (largest t-1 eigenvectors) of the original matrix."""
    osc = []
    for i in range(n):
        s = 0
        for j in range(n - t + 1, n):
            overlap = np.dot(v_pm[:, i], v_orig[:, j])
            s += overlap ** 2
        osc.append(s)
    return osc

## 3. Participation Ratio — Single Window Example
PR computed for the Power-Mapped eigenvectors of a representative window.

In [ ]:
window_idx = 110
fecha = d[window_idx * 22]

PMvectors = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (window_idx, q))
PR = participation_ratio(PMvectors)

plt.plot(PR)
plt.xlabel(r'Eigenvector index $j$')
plt.ylabel(r'$PR(V_{PM}^j)$')
plt.title(fecha, fontsize=14)
plt.tight_layout()
plt.savefig('../figures/participation_ratio_single_window.png', bbox_inches='tight')
plt.show()

## 4. Participation Ratio — Average Across All Windows
Average PR profile across all 262 rolling windows.

In [ ]:
PR_all = []
for i in range(numVentana):
    PMvectors = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (i, q))
    PR_all.append(participation_ratio(PMvectors))
PR_all = np.asarray(PR_all)

plt.plot(PR_all.mean(axis=0))
plt.xlabel(r'Eigenvector index $j$')
plt.ylabel('Mean Participation Ratio')
plt.title('Average PR across all windows (1992-2014)')
plt.tight_layout()
plt.savefig('../figures/participation_ratio_average.png', bbox_inches='tight')
plt.show()

## 5. Overlap Spectral Contribution (OSC) — Single Window
Compares Power-Mapped eigenvectors against the original (non-PM) eigenvectors for the same window, to see how much Power Mapping perturbs the eigenvector structure.

In [ ]:
vectors = np.loadtxt('../data/evector%d.dat' % window_idx)
PMvectors = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (window_idx, q))

OSC = overlap_spectral_contribution(PMvectors, vectors, N, T)

plt.semilogy(OSC)
plt.xlabel(r'$V_{PM}^j$')
plt.ylabel(r'$OSC(V_{PM}^j)$')
plt.title(fecha, fontsize=14)
plt.tight_layout()
plt.savefig('../figures/OSC_single_window.png', bbox_inches='tight')
plt.show()

## 6. OSC of the Top Eigenvectors Across Time
Sums OSC over the top N-T+1 eigenvectors (the signal eigenvectors) for each window, tracking how much Power Mapping perturbs the genuine market structure over time. Spikes are expected during periods of market stress, when the background correlation <C_ij> rises sharply.

In [ ]:
sumaOSC = []
for i in range(0, numVentana - 1, 2):
    vectors_i = np.loadtxt('../data/evector%d.dat' % i)
    PMvectors_i = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (i, q))
    OSC_i = np.asarray(overlap_spectral_contribution(PMvectors_i, vectors_i, N, T))
    sumaOSC.append(np.sum(OSC_i[:N-T+1]))

plt.plot(sumaOSC)
plt.xlabel('Window')
plt.ylabel(r'$\sum OSC$ (signal eigenvectors)')
plt.title('Power Mapping perturbation over time')
plt.tight_layout()
plt.savefig('../figures/OSC_over_time.png', bbox_inches='tight')
plt.show()

In [ ]:
# Map GICS codes to sector abbreviations
gicname = g[:]
replacements = {
    '10': 'E',  '15': 'M',  '20': 'I',  '25': 'CD',
    '30': 'CS', '35': 'HC', '40': 'F',  '45': 'IT',
    '50': 'TS', '55': 'U'
}
for i in range(293):
    for code, name in replacements.items():
        gicname[i] = gicname[i].replace(code, name)

num = 15
sector2 = gicname[:293:num]
indice = np.arange(0, 293, num)
print('Sector labels:', sector2)

## 7. Intensity — Single Eigenvector
Intensity of eigenvector k for stock l: I_kl = u_kl^2

Comparing Power-Mapped (PM) vs original eigenvectors reveals how Power Mapping
redistributes eigenvector weight across stocks and sectors.

In [ ]:
window_idx = 223
numvec = 249
fecha = d[22 + (window_idx * 22)]

vv1 = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (window_idx, q))
vv  = np.loadtxt('../data/evector%d.dat' % window_idx)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vv1[:, numvec]**2, color='red',   label='VP (PM) %d' % numvec)
ax.plot(vv[:, numvec]**2,  color='black',  label='V %d' % numvec, alpha=0.7)
ax.set_xticks(indice)
ax.set_xticklabels(sector2, rotation=30)
ax.set_xlabel('Stock (sector label every %d stocks)' % num)
ax.set_ylabel('Intensity u_kl^2')
ax.set_title(fecha, fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/intensity_single_eigenvector.png', bbox_inches='tight')
plt.show()

## 8. Intensity — Top Eigenvectors Across All Windows
Saves intensity figures for the top signal eigenvectors (indices N-T+1 to N)
for every other window, comparing PM vs original.
These figures reveal which sectors dominate each market mode over time.

In [ ]:
import os
os.makedirs('../figures/intensity', exist_ok=True)

for aaa in range(0, numVentana, 2):
    fecha = d[22 + (aaa * 22)]
    vv1 = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (aaa, q))
    vv  = np.loadtxt('../data/evector%d.dat' % aaa)
    for numvec in range(N - T + 1, N):
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(vv1[:, numvec]**2, color='red',   label='VP (PM) %d' % numvec)
        ax.plot(vv[:, numvec]**2,  color='black',  label='V %d' % numvec, alpha=0.7)
        ax.set_xticks(indice)
        ax.set_xticklabels(sector2, rotation=30)
        ax.set_xlabel('Stock (sector label every %d stocks)' % num)
        ax.set_ylabel('Intensity u_kl^2')
        ax.set_title(fecha, fontsize=14, fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.savefig('../figures/intensity/vector%d_%d_q%.2f.png' % (aaa, numvec, q),
                    bbox_inches='tight')
        plt.clf()
        plt.close()
    if aaa % 50 == 0:
        print(f'Window {aaa}/{numVentana} done')

print('All intensity figures saved to ../figures/intensity/')

## 7. Intensity — Which Stocks Dominate Each Mode?
Intensity of stock $i$ in eigenvector $k$:

$$I_{ki} = N \cdot u_{ki}^2$$

- $I_{ki} > 1$: stock contributes more than average to mode $k$
- $I_{ki} < 1$: stock contributes less than average

Comparing PM vs non-PM eigenvectors shows how Power Mapping redistributes stock contributions within each mode.

In [ ]:
# GICS sector labels per stock
gicname = g[:]
replacements = {
    '10':'E', '15':'M', '20':'I', '25':'CD', '30':'CS',
    '35':'HC', '40':'F', '45':'IT', '50':'TS', '55':'U'
}
for i in range(N):
    for code, label in replacements.items():
        gicname[i] = gicname[i].replace(code, label)

num = 15
sector2 = gicname[:N:num]
indice = np.arange(0, N, num)

# Single window example
window_idx = 110
fecha = d[22 + window_idx * 22]
numvec = 249

vv1 = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (window_idx, q))
vv  = np.loadtxt('../data/evector%d.dat' % window_idx)

intensity_PM   = N * vv1[:, numvec]**2
intensity_orig = N * vv[:, numvec]**2

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(intensity_PM,   color='red',   label='VP PM')
ax.plot(intensity_orig, color='black', label='V original')
ax.axhline(1, color='gray', linestyle='--', linewidth=0.8)
ax.set_xticks(indice)
ax.set_xticklabels(sector2, rotation=30)
ax.set_xlabel('Stock', fontsize=13)
ax.set_ylabel('Intensity', fontsize=13)
ax.set_title(fecha, fontsize=14, fontweight='bold')
ax.legend()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
plt.savefig('../figures/intensity_single_window.png', bbox_inches='tight')
plt.show()

In [ ]:
import os
os.makedirs('../figures/intensity', exist_ok=True)

for aaa in range(0, numVentana, 2):
    fecha = d[22 + aaa * 22]
    vv1 = np.loadtxt('../data/evectorPM%dq%.2f.dat' % (aaa, q))
    vv  = np.loadtxt('../data/evector%d.dat' % aaa)
    for numvec in range(N - T + 1, N):
        intensity_PM   = N * vv1[:, numvec]**2
        intensity_orig = N * vv[:, numvec]**2

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(intensity_PM,   color='red',   label='VP PM')
        ax.plot(intensity_orig, color='black', label='V original')
        ax.axhline(1, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xticks(indice)
        ax.set_xticklabels(sector2, rotation=30)
        ax.set_xlabel('Stock', fontsize=13)
        ax.set_ylabel('Intensity', fontsize=13)
        ax.set_title(fecha, fontsize=14, fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.savefig('../figures/intensity/vector%d_%d_q%.2f.png' % (aaa, numvec, q),
                    bbox_inches='tight')
        plt.clf()
        plt.close()
    if aaa % 50 == 0:
        print(f'Window {aaa}/{numVentana} done')

print('All intensity figures saved.')

## Summary
- **PR** shows the eigenvector structure: the dominant market mode has high PR (most stocks contribute), while sector modes have low PR (few stocks dominate).
- **OSC** confirms Power Mapping has minimal effect on genuine signal eigenvectors while reshaping the noise subspace as intended.
- **Intensity** reveals which specific stocks and sectors drive each market mode, and how Power Mapping redistributes eigenvector weight across the spectrum.